In [1]:
BRANCH = "development"   # instead of test-v0.5.6
REPO_URL = "git@github.com:St1p42/sglang.git"
REPO_DIR = "/content/sglang"

In [2]:
# Generate a public key for SSH - add the printed result to your GitHub account
%%bash
set -e

mkdir -p /root/.ssh
chmod 700 /root/.ssh

if [ ! -f /root/.ssh/id_ed25519 ]; then
  ssh-keygen -t ed25519 -C "colab" -f /root/.ssh/id_ed25519 -N ""
fi

ssh-keyscan github.com >> /root/.ssh/known_hosts 2>/dev/null
chmod 600 /root/.ssh/known_hosts

echo "Public key:"
cat /root/.ssh/id_ed25519.pub

Generating public/private ed25519 key pair.
Your identification has been saved in /root/.ssh/id_ed25519
Your public key has been saved in /root/.ssh/id_ed25519.pub
The key fingerprint is:
SHA256:ZvWIrGbMl0zjo4+nEYhMO/kmknxmS6DQdXAkFCBGFIU colab
The key's randomart image is:
+--[ED25519 256]--+
|+==+=oo          |
|.E   +           |
|  . . .   .      |
| + = o . o o     |
|..B . . S . .    |
|+..o o O o       |
|+..=o B *        |
| .=o.o =..       |
|   .  ++.        |
+----[SHA256]-----+
Public key:
ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIMo5wdstj/Aq75Yvsm24D3Z+EBl06qBH1KHxLzfHrxRI colab


In [3]:
# Ssh test
%%bash
set -e
ssh -T git@github.com || true

Hi StephenKhoaUmass! You've successfully authenticated, but GitHub does not provide shell access.


In [4]:
import os
os.environ["BRANCH"] = BRANCH
os.environ["REPO_URL"] = REPO_URL
os.environ["REPO_DIR"] = REPO_DIR
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'

In [5]:
import os

# Clone if the repo doesn't exist yet
if not os.path.exists(f"{REPO_DIR}/.git"):
    !git clone {REPO_URL} {REPO_DIR}

# Change directory and update repo
%cd {REPO_DIR}
!git fetch origin
!git checkout {BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log --oneline -1

Cloning into '/content/sglang'...
remote: Enumerating objects: 82652, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 82652 (delta 13), reused 9 (delta 9), pack-reused 82620 (from 2)
Receiving objects: 100% (82652/82652), 51.21 MiB | 7.70 MiB/s, done.
Resolving deltas: 100% (59309/59309), done.
/content/sglang
Branch 'development' set up to track remote branch 'development' from 'origin'.
Switched to a new branch 'development'
From github.com:St1p42/sglang
 * branch                development -> FETCH_HEAD
Already up to date.
development
aebf4c157 (HEAD -> development, origin/development) Add files via upload


In [6]:
# See GPU/CPU/RAM info
%%bash
echo "=== GPU Info ==="
nvidia-smi || echo "No GPU attached or nvidia-smi failed"

echo -e "\n=== CPU Info ==="
lscpu | grep 'Model name\|Socket(s)\|Core(s) per socket\|Thread(s) per core'

echo -e "\n=== RAM Info ==="
free -h

=== GPU Info ===
Sat May  2 02:17:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   36C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

In [7]:
# Install
%%bash
set -e
cd /content/sglang

python -m pip uninstall -y sglang sgl-kernel flashinfer-python || true
python -m pip install -e python nvidia-ml-py

Obtaining file:///content/sglang/python
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 9.8 MB/s eta 0:00:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


In [8]:
!pip install transformers dspy-ai==2.5.43 openai==2.6.1 backoff ujson magicattr optuna rouge-score

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of litellm to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of litellm to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.5/344.5 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

In [9]:
# You should see something like:
# torch: 2.9.1+cu128
# sglang version: 0.5.6
# sglang path: ['/content/sglang/python/sglang']
# sglang spec origin: /content/sglang/python/sglang/__init__.py
# sgl_kernel: 0.3.18.post2
# cuda: True NVIDIA L4
import sys

# Make sure Python imports the actual package in your repo
sys.path = [p for p in sys.path if p != "/content/sglang"]
sys.path.insert(0, "/content/sglang/python")

import sglang, torch, sgl_kernel
from importlib.metadata import version

print("torch:", torch.__version__)
print("sglang version:", version("sglang"))
print("sglang path:", list(sglang.__path__))
print("sglang spec origin:", sglang.__spec__.origin)
print("sgl_kernel:", getattr(sgl_kernel, "__version__", "ok"))
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0))

torch: 2.9.1+cu128
sglang version: 0.5.6
sglang path: ['/content/sglang/python/sglang']
sglang spec origin: /content/sglang/python/sglang/__init__.py
sgl_kernel: 0.3.18.post2
cuda: True NVIDIA L4


In [10]:
# Launch the server (change the model if needed)
%%bash
cd /content/sglang
nohup python -m sglang.launch_server \
  --model-path Qwen/Qwen2.5-7B-Instruct \
  --port 30000 \
  --radix-cache-impl custom \
  --schedule-policy fcfs \
  > /content/sglang_server.log 2>&1 &

echo $! > /content/sglang_server.pid
sleep 20
tail -n 50 /content/sglang_server.log

2026-05-02 02:20:18.691108: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 02:20:18.760921: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[2026-05-02 02:20:21] INFO utils.py:164: NumExpr defaulting to 12 threads.


In [15]:
# Check readiness
%%bash
curl -s http://127.0.0.1:30000/v1/models || true
echo
tail -n 80 /content/sglang_server.log

{"object":"list","data":[{"id":"Qwen/Qwen2.5-7B-Instruct","object":"model","created":1777688604,"owned_by":"sglang","root":"Qwen/Qwen2.5-7B-Instruct","parent":null,"max_model_len":32768}]}
[2026-05-02 02:21:21] Using custom radix attention implementation.
we are in custom radix attention
[2026-05-02 02:21:21] Using custom radix attention implementation.
we are in custom radix attention
[2026-05-02 02:21:21] Using custom radix attention implementation.
we are in custom radix attention
[2026-05-02 02:21:21] Using custom radix attention implementation.
we are in custom radix attention
[2026-05-02 02:21:21] Using custom radix attention implementation.
we are in custom radix attention
[2026-05-02 02:21:21] Using custom radix attention implementation.
we are in custom radix attention
[2026-05-02 02:21:21] Using custom radix attention implementation.
we are in custom radix attention
[2026-05-02 02:21:21] Using custom radix attention implementation.
we are in custom radix attention
[2026-05-02

In [ ]:
# In your result JSONL, check the ratio
cached = [0, 0, 4, 4, 4, 4, 25, 4, 5, 25, 66, 5, 4, 4, 66, 4]
input_lens = [110, 109, 104, 136, 117, 111, 108, 111] * 2
print([c/l for c,l in zip(cached, input_lens)])

[0.0, 0.0, 0.038461538461538464, 0.029411764705882353, 0.03418803418803419, 0.036036036036036036, 0.23148148148148148, 0.036036036036036036, 0.045454545454545456, 0.22935779816513763, 0.6346153846153846, 0.03676470588235294, 0.03418803418803419, 0.036036036036036036, 0.6111111111111112, 0.036036036036036036]


In [12]:
# ── Stubs for missing SGLang internals ────────────────────────────────────
with open('/content/sglang/python/sglang/srt/compilation/compilation_config.py', 'a') as f:
    f.write('''
def register_split_op():
    def decorator(fn):
        return fn
    return decorator
''')

with open('/content/sglang/python/sglang/srt/compilation/piecewise_context_manager.py', 'w') as f:
    f.write('''from contextlib import contextmanager

def get_forward_context():
    return None

def is_in_piecewise_cuda_graph():
    return False

@contextmanager
def enable_piecewise_cuda_graph():
    yield

def set_forward_context(*args, **kwargs):
    pass
''')

with open('/content/sglang/python/sglang/srt/utils/custom_op.py', 'w') as f:
    f.write('''
def register_custom_op(mutates_args=None):
    def decorator(fn):
        return fn
    return decorator
''')

# ── Clear bytecode cache ──────────────────────────────────────────────────
os.system('find /content/sglang/python/sglang/srt/ -name "*.pyc" -delete')
os.system('find /content/sglang/python/sglang/srt/ -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null; echo "pyc cleared"')

# ── Verify ────────────────────────────────────────────────────────────────
import subprocess
checks = [
    ('register_split_op',  'grep -c "def register_split_op" /content/sglang/python/sglang/srt/compilation/compilation_config.py'),
    ('get_forward_context','grep -c "def get_forward_context" /content/sglang/python/sglang/srt/compilation/piecewise_context_manager.py'),
    ('register_custom_op', 'grep -c "def register_custom_op" /content/sglang/python/sglang/srt/utils/custom_op.py'),
]
for label, cmd in checks:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    count = result.stdout.strip()
    status = "✓" if count and int(count) > 0 else "✗ MISSING"
    print(f"{status}  {label}")

print("\nSetup complete. Safe to start SGLang server.")

✓  register_split_op
✓  get_forward_context
✓  register_custom_op

Setup complete. Safe to start SGLang server.


In [ ]:
!pkill -f "sglang.launch_server"
!pkill -f "sglang"
!sleep 3

In [ ]:
!ps aux | grep sglang | grep -v grep

In [ ]:
with open('/content/sglang/python/sglang/srt/managers/simple_lpm_scheduler.py', 'r') as f:
    src = f.read()

old = """            r.prefix_indices = match_result.device_indices
            r.last_node = match_result.last_device_node
            r.last_host_node = match_result.last_host_node
            r.host_hit_length = match_result.host_hit_length"""

new = """            r.prefix_indices = match_result.device_indices
            r.last_node = match_result.last_device_node
            r.last_host_node = match_result.last_host_node
            r.host_hit_length = match_result.host_hit_length
            # Protect the matched node so it cannot be evicted while this
            # request is in flight. Matches the dec_lock_ref in
            # cache_finished_req. Without this, lock_ref stays 0 and the
            # node is immediately evictable, collapsing cache hit rate.
            if r.last_node is not None:
                self.tree_cache.inc_lock_ref(r.last_node)"""

if old in src:
    src = src.replace(old, new)
    with open('/content/sglang/python/sglang/srt/managers/simple_lpm_scheduler.py', 'w') as f:
        f.write(src)
    print("Patch applied")
else:
    print("Pattern not found — check whitespace:")
    idx = src.find("r.prefix_indices = match_result.device_indices")
    print(repr(src[idx:idx+300]))

Patch applied


In [13]:
# Patch so that we don't rely on the vLLM dependency just for the tokenizer
%%bash
set -e
cd /content/sglang/benchmark/rag

python - <<'PY'
from pathlib import Path

p = Path("bench_dspy_rag_serving.py")
s = p.read_text()

s = s.replace(
    "from vllm.transformers_utils.tokenizer import get_tokenizer",
    "from transformers import AutoTokenizer",
)
s = s.replace(
    "tokenizer = get_tokenizer(args.tokenizer, trust_remote_code=args.trust_remote_code)",
    "tokenizer = AutoTokenizer.from_pretrained(args.tokenizer, trust_remote_code=args.trust_remote_code)",
)

p.write_text(s)
print("patched bench_dspy_rag_serving.py")
PY

patched bench_dspy_rag_serving.py


In [16]:
# Cell 2: Warmup + flush
%%bash
set -e
cd /content/sglang/benchmark/rag

# Warmup
TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --dataset-file workloads/workload_msmarco_interleaved_24x20.jsonl \
  --max-requests 64 \
  --parallel 8 \
  > /tmp/bench_warmup.log
echo "Warmup done"

sleep 10
curl -s -X POST http://127.0.0.1:30000/flush_cache
echo "Cache flushed"
sleep 2

# Real run
TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --dataset-file workloads/workload_msmarco_interleaved_24x20.jsonl \
  --parallel 64 \
  --result-file /content/sglang_server.log

Warmup done
Cache flushed.
Please check backend logs for more details. (When there are running or waiting requests, the operation will not be performed.)
Cache flushed
[dataset] Loaded 480 prebuilt prompts from workloads/workload_msmarco_interleaved_24x20.jsonl
[dataset] JSONL row order is preserved when creating async tasks.

================ DSPy RAG Benchmark Result =================
Request Throughput (req/s)                    18.57     
Input Token Throughput (tok/s)                13438.13  
Output Token Throughput (tok/s)               594.18    
Total Token Throughput (tok/s)                14032.31  
Concurrency                                   64.00     
Cache Hit Rate                                0.938743  
------------------------- Latency --------------------------
Mean E2E Latency (ms)                         3234.18   
Mean TTFT (ms)                                799.40    
Mean TPOT (ms)                                78.54     
----------------------- Tail Latency

/usr/local/lib/python3.12/dist-packages/dsp/modules/google.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
/usr/local/lib/python3.12/dist-packages/dsp/modules/google.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [20]:
import json
with open('/content/results_msmarco_p64_raw.json') as f:
    r = json.load(f)
print(r['details']['cached_tokens'][:24])

[0, 0, 0, 0, 2, 3, 2, 2, 2, 2, 3, 2, 3, 2, 2, 2, 3, 2, 2, 3, 3, 2, 2, 3]


In [17]:
%%bash
set -e
cd /content/sglang
git fetch origin
git pull origin $BRANCH
git log --oneline -1

Updating d30071d5a..aebf4c157
Fast-forward
 .../rag/workloads/workload_msmarco_interleaved_24x20.jsonl | 480 +++++++++++++++++++++
 1 file changed, 480 insertions(+)
 create mode 100644 benchmark/rag/workloads/workload_msmarco_interleaved_24x20.jsonl
aebf4c157 Add files via upload


From github.com:St1p42/sglang
   d30071d5a..aebf4c157  development -> origin/development
From github.com:St1p42/sglang
 * branch                development -> FETCH_HEAD


In [36]:
%%bash
set -e

curl -s -X POST http://127.0.0.1:30000/flush_cache
echo "Cache flushed"

CalledProcessError: Command 'b'set -e\n\ncurl -s -X POST http://127.0.0.1:30000/flush_cache\necho "Cache flushed"\n'' returned non-zero exit status 7.

In [ ]:
# Run multi-turn chat benchmark
%%bash
set -e
cd /content/sglang/benchmark/rag

# Warmup
TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --num-questions 20 \
  --parallel 1 \
  > /tmp/bench_warmup.log
echo "Warmup done"

# Wait for all warmup requests to fully complete
sleep 10

# Flush cache
curl -s -X POST http://127.0.0.1:30000/flush_cache
echo "Cache flushed"

TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --num-questions 200 \
  --parallel 4 \
  --result-file /content/sglang_server.log

bash: line 2: cd: /content/sglang/benchmark/rag: No such file or directory


CalledProcessError: Command 'b'set -e\ncd /content/sglang/benchmark/rag\n\n# Warmup\nTF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py \\\n  --model Qwen/Qwen2.5-7B-Instruct \\\n  --num-questions 20 \\\n  --parallel 1 \\\n  > /tmp/bench_warmup.log\necho "Warmup done"\n\n# Wait for all warmup requests to fully complete\nsleep 10\n\n# Flush cache\ncurl -s -X POST http://127.0.0.1:30000/flush_cache\necho "Cache flushed"\n\nTF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py \\\n  --model Qwen/Qwen2.5-7B-Instruct \\\n  --num-questions 200 \\\n  --parallel 4 \\\n  --result-file /content/sglang_server.log\n'' returned non-zero exit status 1.

In [ ]:
import json
with open('/content/sglang_server.log') as f:
    lines = [l.strip() for l in f if l.strip()]
r = json.loads(lines[-1])
print(r['details']['cached_tokens'][:16])

[0, 4, 25, 5, 4, 4, 66, 4, 109, 108, 103, 135, 116, 110, 107, 110]


In [ ]:
# Try with 1, 4, and 8 parallel conversations (with a total of 20)
%%bash
set -e
cd /content/sglang/benchmark/rag
TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --num-questions 200 \
  --parallel 4 \
  --result-file /content/sglang_server.log

# Wait for all warmup requests to fully complete
sleep 10

# Flush cache
curl -s -X POST http://127.0.0.1:30000/flush_cache
echo "Cache flushed"

TF_CPP_MIN_LOG_LEVEL=3 python bench_dspy_rag_serving.py \
  --model Qwen/Qwen2.5-7B-Instruct \
  --num-questions 200 \
  --parallel 8 \
  --result-file /content/sglang_server.log

[retriever] Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 on cuda
[retriever] No external endpoint — using local EmbeddingRetriever

================ DSPy RAG Benchmark Result =================
Request Throughput (req/s)                    2.04      
Input Token Throughput (tok/s)                206.38    
Output Token Throughput (tok/s)               65.39     
Total Token Throughput (tok/s)                271.77    
Concurrency                                   4.00      
Cache Hit Rate                                0.762178  
------------------------- Latency --------------------------
Mean E2E Latency (ms)                         1929.99   
Mean TTFT (ms)                                122.23    
Mean TPOT (ms)                                58.31     
----------------------- Tail Latency -----------------------
  TTFT                 p50=    103.62  p95=    190.39  p99=    670.84
  E2E Latency          p50=   1911.00  p95=   1999.91  p99=   2476.30
-------------

/usr/local/lib/python3.12/dist-packages/dsp/modules/google.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
/usr/local/lib/python3.12/dist-packages/dsp/modules/google.py:11: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [19]:
%%bash
tail -200 /content/sglang_server.log | grep "\[SCHED TRACE\]" || echo "No SCHED TRACE logs found"

No SCHED TRACE logs found


In [22]:
!ls

3rdparty  benchmark	      docs	Makefile   scripts     test
690AB	  CODE_OF_CONDUCT.md  examples	python	   sgl-kernel
assets	  docker	      LICENSE	README.md  sgl-router


In [34]:
# 1. Kill ALL sglang servers
!pkill -f "sglang.launch_server" || true
!kill $(cat /content/sglang_server.pid || true) || true

# 2. Verify nothing on port 30000
!lsof -i :30000 || echo "Port 30000 is free"

^C
/bin/bash: line 1: kill: (13553) - No such process
COMMAND  PID USER   FD   TYPE DEVICE SIZE/OFF NODE NAME
python3 7139 root   56u  IPv4 206744      0t0  TCP localhost:30000 (LISTEN)


In [35]:
%%bash
# Kill the process actually on port 30000
kill -TERM 7139

# Verify it's gone (wait 3s)
sleep 3
lsof -i :30000 || echo "Port 30000 is now free ✅"

Port 30000 is now free ✅


bash: line 2: kill: (7139) - No such process


In [ ]:
tail -f /content/sglang_server.log | grep "\[CUSTOM SCHED\]"